# Get racmo downscaled fields and upload to scratch bucket

This downloads all the netcdfs supplied ot me by Brice Noel in an email to j.kingslake@columbia.edu on May 27 2026 and puts them in my cryocloud stratch bucket. 

Main code developed using Chatgpt.

- J.Kingslake, May 27, 2026

In [1]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=2,
    memory_limit="8GB",
)

client = Client(cluster)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Workers: 4
Total threads: 8,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45205,Workers: 4
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Total threads: 8
Started: Just now,Total memory: 29.80 GiB
Comm: tcp://127.0.0.1:44727,Total threads: 2
Dashboard: /user/jkingslake/load%20NCs/proxy/42529/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:32925,


In [11]:
client.shutdown()

In [2]:
import requests
import xarray as xr
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from tqdm.auto import tqdm

import s3fs

fs = s3fs.S3FileSystem()

BASE_URL = "http://climato.be/ftp/climato/bnoel/Share-public/.Kingslake/Daily-2km/"

# DATASETS = ["snowmelt", "ff10m", "precip", "t2m"]
DATASETS = ["ff10m"]

S3_BASE = "s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr"

TMP_DIR = Path("tmp_nc")
TMP_DIR.mkdir(exist_ok=True)


def list_nc_files(url):
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    return sorted(
        a["href"]
        for a in soup.find_all("a", href=True)
        if a["href"].endswith(".nc")
    )


for dataset in DATASETS:
    dataset_url = urljoin(BASE_URL, dataset + "/")
    files = list_nc_files(dataset_url)

    print(f"{dataset}: {len(files)} files")

    for filename in tqdm(files):
        source_url = urljoin(dataset_url, filename)
        local_nc = TMP_DIR / filename

        zarr_name = Path(filename).stem + ".zarr"
        zarr_path = f"{S3_BASE}/{dataset}/{zarr_name}"

        if fs.exists(zarr_path + "/.zmetadata"):

            print(f"Skipping existing zarr {zarr_path}")
        
            continue
        
        if local_nc.exists():
            print(f"Using existing local file {local_nc}")
        else:
            print(f"Downloading {filename}")
        
            r = requests.get(source_url)
            r.raise_for_status()

            local_nc.write_bytes(r.content)

        print(f"Opening {filename}")
        ds = xr.open_dataset(local_nc, engine="h5netcdf", chunks="auto")
        
        try:
            print(f"Writing {zarr_path}")
            ds.to_zarr(
                zarr_path,
                mode="w",
                consolidated=True,
                zarr_format=2,
            )
        except Exception as e:

            print("failed .load()",local_nc, e)
            continue
        
        ds.close()
        local_nc.unlink()

print("Done")

ff10m: 188 files


  0%|          | 0/188 [00:00<?, ?it/s]

Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1979_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1979_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1979_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1980_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1980_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1980_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scr

Exception in thread Thread-5:
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tqdm/_monitor.py", line 84, in run
    instance.refresh(nolock=True)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tqdm/std.py", line 1347, in refresh
    self.display()
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tqdm/notebook.py", line 171, in display
    rtext.value = right
    ^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/traitlets/traitlets.py", line 716, in __set__
    self.set(obj, value)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/traitlets/traitlets.py", line 706, in set
    obj._notify_trait(self.name, old_value, new_value)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/traitlets/traitlets.py", line 1513, in _notify_trait
    self.notify_change(
  Fil

Opening ff10m.2000_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2000_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2000_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2000_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2001_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2001_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr


...

failed .load() tmp_nc/ff10m.2015_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc Can't synchronously read data (inflate() failed)
Opening ff10m.2015_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2015_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2015_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2015_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2016_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2016_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2016_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2016_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2016_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff

2026-05-28 10:24:26,404 - distributed.worker - ERROR - Compute Failed
Key:       ('where-store-map-60da1eb540ba61c21d19e41f386accfa', 18, 0, 0)
State:     executing
Task:  <Task ('where-store-map-60da1eb540ba61c21d19e41f386accfa', 18, 0, 0) _execute_subgraph(...)>
Exception: 'OSError("Can\'t synchronously read data (inflate() failed)")'
Traceback: '  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/array/core.py", line 129, in getter\n    c = np.asarray(c)\n        ^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py", line 688, in __array__\n    return np.asarray(self.get_duck_array(), dtype=dtype)\n                      ^^^^^^^^^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py", line 691, in get_duck_array\n    return self.array.get_duck_array()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py"

failed .load() tmp_nc/ff10m.2019_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc Can't synchronously read data (inflate() failed)
Opening ff10m.2019_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2019_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2019_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2019_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2020_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2020_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2020_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2020_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr


2026-05-28 10:33:23,850 - distributed.worker - ERROR - Compute Failed
Key:       ('where-store-map-3a5b0aceefa8b6e4e5fc62476ab36bfb', 7, 0, 0)
State:     executing
Task:  <Task ('where-store-map-3a5b0aceefa8b6e4e5fc62476ab36bfb', 7, 0, 0) _execute_subgraph(...)>
Exception: 'OSError("Can\'t synchronously read data (inflate() failed)")'
Traceback: '  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/array/core.py", line 129, in getter\n    c = np.asarray(c)\n        ^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py", line 688, in __array__\n    return np.asarray(self.get_duck_array(), dtype=dtype)\n                      ^^^^^^^^^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py", line 691, in get_duck_array\n    return self.array.get_duck_array()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/core/indexing.py", 

failed .load() tmp_nc/ff10m.2020_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc Can't synchronously read data (inflate() failed)
Opening ff10m.2020_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2020_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2020_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2020_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2021_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2021_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2021_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.2021_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Opening ff10m.2021_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.nc
Writing s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff